# MK7 Router v1.2 — تدريب على Colab

قاعدة مجمدة `Qwen2.5-0.5B` + تدريب رأس Router فقط على داتاسيت `mk7-v0.2-real` الرياضيات (343 سجلًا).

**قبل التشغيل:** Runtime → Change runtime type → **T4 GPU**.

النواتج تُحفَظ في Google Drive تحت `MK7/router-v1.2.0-real` حتى لا تضيع عند قطع الجلسة.

In [ ]:
# 1) التحقق من الـGPU
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU fallback')
assert torch.cuda.is_available(), 'فعّل GPU من Runtime > Change runtime type'

In [ ]:
# 2) ربط Drive (لحفظ النواتج) + جلب المستودع والفرع
from google.colab import drive
drive.mount('/content/drive')
!git clone --depth 1 -b data-tooling-v0.1 https://github.com/saskw2010/ai-sovereignty-colibri-router-lab.git
%cd ai-sovereignty-colibri-router-lab

In [ ]:
# 3) التدريب (القاعدة تتحمل من HuggingFace مباشرة ~1GB)
!python experiments/train_mk7_router_v1_2.py \
  --dataset data/mk7-v0.2-real/batches/math-v0.1/mk7-v0.2-real-dataset.jsonl \
  --out "/content/drive/MyDrive/MK7/router-v1.2.0-real" \
  --base Qwen/Qwen2.5-0.5B \
  --steps 500 --seed 20260905

In [ ]:
# 4) عرض التقرير النهائي
import json
r = json.load(open('/content/drive/MyDrive/MK7/router-v1.2.0-real/result.json', encoding='utf-8'))
print('أرضية الـuntrained router (validation top1):', r['untrained_router_validation_top1'])
print('بعد التدريب  — validation:', r['validation'])
print('               challenge  :', r['challenge'], '  (v1.0 القديم: 0.857)')
print('best_step:', r['best_step'], '| loss:', round(r['initial_loss'],2), '->', round(r['final_loss'],3))
print('حوكمة: base_frozen =', r['base_frozen'], '| held_out_used =', r['held_out_used'])

## بعد التشغيل
- انسخ أرقام `result.json` للمراجعة.
- المقارنة المستهدفة: تجاوز أرضية الـuntrained router بفارق واضح على validation، وتحسين 85.7% على challenge.
- `held_out` لم يُستخدم — محجوز للحكم النهائي وفق البوابات.